# Numerical integration and convergence order

*maths · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/numerical_integration)

The trapezoid rule, Simpson's rule and Gauss–Legendre quadrature are all a weighted sum $\sum_k w_k f(x_k)$, one einsum over many integrands at once. The Gauss nodes and weights come from an eigenvalue problem (Golub–Welsch). On log–log axes the errors fall with slopes 2 and 4, or exponentially, until a kink or a singularity takes the high order away.

**Vector form**

$$\int_{-1}^{1} f(x)\,dx \approx \sum_{k} w_k\,f(x_k),\qquad |\text{error}| \propto N^{-p}$$

**Index (tensor) form**

$$I_f = w_k\,F_{fk},\qquad F_{fk} = f_f(x_k),\qquad J_{kl}\,V_{lm} = x_m\,V_{km},\qquad w_m = 2\,V_{0m}V_{0m}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_k,\ w_k$ | `trapezoid(n, dev), simpson(n, dev)` | (N+1,), (N+1,) | equally spaced nodes; weights $h(\tfrac12, 1, \dots, 1, \tfrac12)$ and $\tfrac h3(1, 4, 2, \dots, 4, 1)$ |
| $J_{kl}$ | `jacobi_matrix(n, dev)` | (N, N) | tridiagonal matrix of the Legendre recurrence, $\beta_k = k/\sqrt{4k^2 - 1}$ beside the diagonal |
| $x_m,\ V_{km}$ | `torch.linalg.eigh(J)` | (N,), (N, N) | eigenvalues are the Gauss–Legendre nodes, eigenvectors in the columns |
| $w_m = 2 V_{0m}^2$ | `2 * V[0] ** 2` | (N,) | Gauss weights from the first component of each eigenvector; they add up to $\int_{-1}^{1} dx = 2$ |
| $F_{fk}$ | `integrands(x, kink, alpha)` | (3, K) | the smooth, kink and endpoint integrands at every node |
| $I_f = w_k F_{fk}$ | `einsum('k,fk->f', w, F)` | (3,) | repeated $k$ ⇒ summed: all three integrals at once |

For a smooth integrand the trapezoid error falls like $h^2 \propto N^{-2}$ and Simpson's like $N^{-4}$: doubling $N$ divides them by 4 and 16. Gauss–Legendre with $N$ nodes integrates every polynomial of degree up to $2N - 1$ exactly, so for an analytic function its error drops exponentially until it reaches float64 round-off near $10^{-16}$. These orders come from Taylor expansions, so they need derivatives. With a kink $|x - a|$ the second derivative is a spike: every rule that straddles the kink makes an $O(h^2)$ error there, so Simpson and Gauss fall back to order 2 (the curve is jagged because where the kink falls inside its panel changes with $N$). With $(1+x)^{\alpha}$ the derivatives blow up at the endpoint: the trapezoid and Simpson rules drop to order $1 + \alpha$ and Gauss, whose nodes cluster towards the ends, to about $2 + 2\alpha$. The cure is to split the interval at the kink or to change variables so the integrand becomes smooth. Try a whole number $\alpha$ (a polynomial) and watch Gauss become exact, or put the kink on a grid node ($a = 0.3$ is one whenever $N$ is a multiple of 20): the trapezoid rule is then exact for the two straight pieces and drops to round-off.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/numerical_integration/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import gauss_legendre, integral


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    kink, alpha = params["kink"], params["alpha"]
    ns = tensors.n_values(params["n_max"])
    print(f"{len(ns)} values of N from {ns[0]} to {ns[-1]}, 3 rules, 3 integrands: {', '.join(tensors.NAMES)}")

    x, w = gauss_legendre(tensors.jacobi_matrix(5, dev))
    print(f"Gauss–Legendre N = 5 nodes {[round(v, 6) for v in x.tolist()]}, weights sum to {w.sum():.15f}")
    print(f"  ∫x⁸ dx = 2/9 = {2 / 9:.15f}, Gauss with 5 nodes gives {integral(w, x[None] ** 8).item():.15f}")

    rules = solve.all_rules(ns, dev)  # rules[method][i] = (nodes, weights)
    err = solve.errors(rules, kink, alpha)  # (3 methods, M, 3 integrands)
    p = solve.orders(ns, err)
    for m, method in enumerate(solve.METHODS):
        cells = [f"{name} {p[m, f]:.2f}" for f, name in enumerate(tensors.NAMES)]
        if m == 2:  # Gauss on a smooth function has no fixed order: the error falls faster than any power of N
            cells[0] = "smooth exp."
        print(f"order p (error ∝ N^-p)  {method:15s} " + "   ".join(cells))
    rho = solve.exponential_rate(ns, err)
    print(f"Gauss–Legendre on the smooth integrand: error ∝ {rho:.2f}^(−N), exponential (no fixed order)")
    print(f"at N = {ns[-1]}: smallest errors " + ", ".join(f"{e:.1e}" for e in err[:, -1].amin(dim=0).tolist()))
    return plot.animate(ns, rules, err, kink, alpha, out_dir / "numerical_integration.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/numerical_integration/equation.py
r"""Maths → tensor translation.

Every quadrature rule replaces the integral by a weighted sum over nodes x_k,

    \int_{-1}^{1} f(x)\,dx \approx \sum_k w_k\,f(x_k),

so with several integrands stacked in F_{fk} = f_f(x_k) all integrals are one einsum, I_f = w_k F_{fk} (k summed).
The rules differ only in their nodes and weights:

- trapezoid, N intervals of width h:  w = h·(½, 1, 1, …, 1, ½)                      error ∝ h²
- Simpson, N even:                    w = h/3·(1, 4, 2, 4, …, 2, 4, 1)              error ∝ h⁴
- Gauss–Legendre, N nodes: the nodes are the eigenvalues of the Jacobi matrix J and the weights come from the
  first component of each eigenvector, J V = V diag(x),  w_k = 2 V_{0k}²  (Golub–Welsch). Exact for every
  polynomial of degree ≤ 2N − 1, so for smooth f the error falls exponentially.
"""

import torch


def integral(w: torch.Tensor, f: torch.Tensor) -> torch.Tensor:
    r"""I_f = w_k F_{fk}   (k summed over the nodes).   w (K,), F (n_f, K) → (n_f,)"""
    return torch.einsum("k,fk->f", w, f)


def trapezoid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Nodes and weights of the composite trapezoid rule with n intervals: (n+1,), (n+1,)."""
    x = torch.linspace(-1, 1, n + 1, dtype=torch.float64, device=device)
    w = torch.full_like(x, 2 / n)
    w[0] = w[-1] = 1 / n  # half weight at the two ends
    return x, w


def simpson(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Nodes and weights of the composite Simpson rule with n (even) intervals: (n+1,), (n+1,)."""
    x = torch.linspace(-1, 1, n + 1, dtype=torch.float64, device=device)
    h = 2 / n
    w = torch.full_like(x, 2 * h / 3)
    w[1::2] = 4 * h / 3  # odd nodes are the midpoints of each pair of intervals
    w[0] = w[-1] = h / 3
    return x, w


def gauss_legendre(jacobi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Golub–Welsch: J V = V diag(x), w_k = μ₀ V_{0k}² with μ₀ = ∫_{-1}^{1} dx = 2.   → nodes (N,), weights (N,)"""
    x, v = torch.linalg.eigh(jacobi)  # eigenvalues in ascending order, eigenvectors in the columns of v
    return x, 2 * v[0] ** 2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/numerical_integration/tensors.py
"""Build the concrete tensors: the test integrands with their exact integrals, the N values and the Jacobi matrix."""

import math

import torch

NAMES = ["smooth", "kink", "endpoint"]


def integrands(x: torch.Tensor, kink: float, alpha: float) -> torch.Tensor:
    """f_f(x_k) for the three test functions on [−1, 1]: (3, K).

    smooth   1 / (1 + 4x²)    analytic on [−1, 1]
    kink     |x − a|          continuous, but its slope jumps at x = a
    endpoint (1 + x)^α        its derivatives blow up at x = −1 (for α not a whole number)
    """
    return torch.stack([1 / (1 + 4 * x**2), (x - kink).abs(), (1 + x).clamp(min=0) ** alpha])


def exact(kink: float, alpha: float) -> torch.Tensor:
    """The exact integrals over [−1, 1]: atan 2, 1 + a², 2^(α+1)/(α+1): (3,)."""
    return torch.tensor([math.atan(2.0), 1 + kink**2, 2 ** (alpha + 1) / (alpha + 1)], dtype=torch.float64)


def n_values(n_max: int) -> list[int]:
    """About 30 even N from 2 to n_max, evenly spread on a log scale (Simpson needs an even number of intervals)."""
    steps = torch.logspace(math.log10(2), math.log10(n_max), 30, dtype=torch.float64)
    return sorted({2 * round(float(n) / 2) for n in steps})


def jacobi_matrix(n: int, device: torch.device) -> torch.Tensor:
    """J_{kl}: the symmetric tridiagonal matrix of the Legendre recurrence, β_k = k / √(4k² − 1) off the diagonal.

    Its eigenvalues are the N Gauss–Legendre nodes: (N, N).
    """
    k = torch.arange(1, n, dtype=torch.float64, device=device)
    beta = k / torch.sqrt(4 * k**2 - 1)  # (N−1,)
    return torch.diag(beta, 1) + torch.diag(beta, -1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/numerical_integration/solve.py
"""Apply the three rules for every N to every integrand, and fit the convergence order of each error curve."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import gauss_legendre, integral, simpson, trapezoid

METHODS = ["trapezoid", "Simpson", "Gauss–Legendre"]
FLOOR = 1e-15  # below this the error is float64 round-off, not the rule


def rule(method: int, n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Nodes and weights of one rule for one N."""
    if method == 0:
        return trapezoid(n, device)
    if method == 1:
        return simpson(n, device)
    return gauss_legendre(tensors.jacobi_matrix(n, device))


def all_rules(ns: list[int], device: torch.device) -> list[list[tuple[torch.Tensor, torch.Tensor]]]:
    """rules[m][i] = (nodes, weights) of method m with N = ns[i]. Built once, reused by the plot."""
    return [[rule(m, n, device) for n in ns] for m in steps(len(METHODS), "rules")]


def errors(rules: list, kink: float, alpha: float) -> torch.Tensor:
    """|I_rule − I_exact| for every method, N and integrand: (3 methods, len(ns), 3 integrands)."""
    device = rules[0][0][0].device
    exact = tensors.exact(kink, alpha).to(device)
    err = torch.empty(len(rules), len(rules[0]), len(exact), dtype=torch.float64, device=device)
    for m, per_n in enumerate(rules):
        for i, (x, w) in enumerate(per_n):
            err[m, i] = (integral(w, tensors.integrands(x, kink, alpha)) - exact).abs()
    return err.clamp(min=FLOOR / 10)


def orders(ns: list[int], err: torch.Tensor) -> torch.Tensor:
    """Fitted p in error ∝ N^(−p) for every method and integrand, from the points with N ≥ 16 above round-off.

    err (3, M, 3) → (3, 3); NaN when fewer than 3 points are usable (Gauss on the smooth integrand hits round-off).
    """
    n = torch.tensor(ns, dtype=torch.float64, device=err.device)
    logn = torch.log(n)
    p = torch.full(err.shape[::2], float("nan"), dtype=torch.float64, device=err.device)
    for m in range(err.shape[0]):
        for f in range(err.shape[2]):
            keep = (n >= 16) & (err[m, :, f] > 1e3 * FLOOR)
            if keep.sum() >= 3:
                x, y = logn[keep] - logn[keep].mean(), torch.log(err[m, keep, f])
                p[m, f] = -(x @ (y - y.mean())) / (x @ x)
    return p


def exponential_rate(ns: list[int], err: torch.Tensor) -> float:
    """ρ in error ∝ ρ^(−N), fitted to the Gauss–Legendre error on the smooth integrand above round-off."""
    n = torch.tensor(ns, dtype=torch.float64)
    e = err[2, :, 0].cpu()
    keep = e > 1e3 * FLOOR
    x, y = n[keep] - n[keep].mean(), torch.log(e[keep])
    return float(torch.exp(-(x @ (y - y.mean())) / (x @ x)))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/numerical_integration/plot.py
"""Animate N growing: the nodes on each integrand (top) and the error curves on log–log axes (bottom)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import tensors
from .solve import FLOOR, METHODS

COLORS = ["#2563eb", "#16a34a", "#dc2626"]  # trapezoid, Simpson, Gauss–Legendre


def animate(ns: list[int], rules: list, err: torch.Tensor, kink: float, alpha: float, path: Path) -> Path:
    """ns (M,), rules[method][i] = (nodes, weights), err (3 methods, M, 3 integrands)."""
    err = err.cpu().numpy()
    n = np.array(ns, dtype=float)
    xs = torch.linspace(-1, 1, 400, dtype=torch.float64)
    fs = tensors.integrands(xs, kink, alpha).numpy()  # (3, 400)
    labels = ["smooth  $1/(1+4x^2)$", f"kink  $|x - {kink:g}|$", f"endpoint  $(1+x)^{{{alpha:g}}}$"]

    fig, axes = plt.subplots(2, 3, figsize=(9, 5.6), gridspec_kw={"height_ratios": [1, 1.4]})
    nodes_t, nodes_g, trails = [], [], []
    for f in range(3):
        top, bottom = axes[0, f], axes[1, f]
        top.plot(xs.numpy(), fs[f], color="#374151", lw=1.4)
        top.fill_between(xs.numpy(), 0, fs[f], color="#93c5fd", alpha=0.35)
        (nt,) = top.plot([], [], "|", color=COLORS[0], ms=8, mew=1)
        (ng,) = top.plot([], [], "o", color=COLORS[2], ms=3.5)
        nodes_t.append(nt)
        nodes_g.append(ng)
        top.set_xlim(-1.05, 1.05)
        top.set_ylim(-0.12, fs[f].max() * 1.12)
        top.set_title(labels[f], fontsize=9)

        for m in range(3):
            (tr,) = bottom.loglog([], [], "o-", color=COLORS[m], ms=2.5, lw=1.3, label=METHODS[m])
            trails.append(tr)
        scale = err[0, 2, f]  # reference slopes, anchored at the trapezoid error at the third N
        bottom.loglog(n, scale * (n / n[2]) ** -2, color="#9ca3af", ls="--", lw=1, label="$N^{-2}$")
        bottom.loglog(n, err[1, 2, f] * (n / n[2]) ** -4, color="#9ca3af", ls=":", lw=1.2, label="$N^{-4}$")
        bottom.set_xlim(n[0] * 0.8, n[-1] * 1.25)
        bottom.set_ylim(FLOOR / 3, max(1.0, err[:, :, f].max() * 3))
        bottom.set_xlabel("N (intervals, or Gauss nodes)", fontsize=8)
        if f == 0:
            bottom.set_ylabel("|error|")
            bottom.legend(fontsize=7, loc="lower left")
    title = fig.suptitle("", fontsize=11)

    def update(i: int):
        x_t = rules[0][i][0].cpu()  # trapezoid / Simpson nodes (the same points)
        x_g = rules[2][i][0].cpu()
        f_g = tensors.integrands(x_g, kink, alpha).numpy()
        for f in range(3):
            nodes_t[f].set_data(x_t.numpy(), np.full(len(x_t), -0.06 * fs[f].max()))
            nodes_g[f].set_data(x_g.numpy(), f_g[f])
            for m in range(3):
                trails[3 * f + m].set_data(n[: i + 1], err[m, : i + 1, f])
        title.set_text(f"N = {ns[i]}:  blue ticks = trapezoid/Simpson points,  red dots = Gauss–Legendre nodes")

    fig.tight_layout()
    frames = frame_indices(len(ns), len(ns))
    return save_gif(fig, update, frames + frames[-1:] * 6, path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 256,
    'kink': 0.577,
    'alpha': 0.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.numerical_integration")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.numerical_integration.main")
problem_dir = Path("problems/maths/numerical_integration")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
